In [23]:
import os
import json
import pandas as pd
from dotenv import load_dotenv, find_dotenv

from doc_chat.llm import create_llm
from doc_chat.rag.citation_retrieval_chain import CitationRetrievalChain
from doc_chat.rag.document_loader import DocumentLoader
from doc_chat.rag.multi_tenant_vector_store import MultiTenantVectorStore

_ = load_dotenv(find_dotenv())


In [2]:
pd_documents = pd.read_csv('../../data/single_topic_rag_evaluation_dataset/processed/documents_processed.csv')
pd_documents

,index,source_url,text,word_count,num_pages
0,0,https://enterthegungeon.fandom.com/wiki/Bullet...,Bullet Kin\nBullet Kin are one of the most com...,1845,4
1,1,https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1a...,---The Paths through the Underground/Underdark...,470,2
2,2,https://bytes-and-nibbles.web.app/bytes/stici-...,Semantic and Textual Inference Chatbot Interfa...,3785,9
3,3,https://github.com/llmware-ai/llmware,llmware\n\nBuilding Enterprise RAG Pipelines w...,3003,8
4,4,https://docs.marimo.io/recipes.html,Recipes\nThis page includes code snippets or “...,1796,5
5,5,https://towardsdatascience.com/how-to-maximize...,How to Maximize Your Impact as a Data Scientis...,2441,6
6,6,https://ec.europa.eu/commission/presscorner/de...,Why do we need to regulate the use of Artifici...,5392,14
7,7,https://bg3.wiki/wiki/The_Emperor,The Emperor is a mind flayer who appears in Ba...,2973,7
8,8,https://whattocook.substack.com/p/so-into-nort...,so into northern spain!\nour magical urban-plu...,2045,5
9,9,https://dmtalkies.com/the-zone-of-interest-end...,‘The Zone Of Interest’ Ending Explained & Film...,2325,5


In [13]:
# Load and split the documents
all_splits = []
for index, row in pd_documents.iterrows():
    doc_path = f"../../data/single_topic_rag_evaluation_dataset/processed/pdf/document_{index}.pdf"
    loader = DocumentLoader(doc_path)
    documents, splits = loader.load_and_split()
    pd_documents.loc[index, 'num_splits'] = int(len(splits))
    all_splits.append(splits)

pd_documents['num_splits'] = pd_documents['num_splits'].astype(int)
pd_documents

,index,source_url,text,word_count,num_pages,num_splits
0,0,https://enterthegungeon.fandom.com/wiki/Bullet...,Bullet Kin\nBullet Kin are one of the most com...,1845,4,13
1,1,https://www.dropbox.com/scl/fi/ljtdg6eaucrbf1a...,---The Paths through the Underground/Underdark...,470,2,4
2,2,https://bytes-and-nibbles.web.app/bytes/stici-...,Semantic and Textual Inference Chatbot Interfa...,3785,9,29
3,3,https://github.com/llmware-ai/llmware,llmware\n\nBuilding Enterprise RAG Pipelines w...,3003,8,24
4,4,https://docs.marimo.io/recipes.html,Recipes\nThis page includes code snippets or “...,1796,5,16
5,5,https://towardsdatascience.com/how-to-maximize...,How to Maximize Your Impact as a Data Scientis...,2441,6,21
6,6,https://ec.europa.eu/commission/presscorner/de...,Why do we need to regulate the use of Artifici...,5392,14,51
7,7,https://bg3.wiki/wiki/The_Emperor,The Emperor is a mind flayer who appears in Ba...,2973,7,26
8,8,https://whattocook.substack.com/p/so-into-nort...,so into northern spain!\nour magical urban-plu...,2045,5,16
9,9,https://dmtalkies.com/the-zone-of-interest-end...,‘The Zone Of Interest’ Ending Explained & Film...,2325,5,17


In [14]:
# create vector store
chroma_persist_directory = '/tmp/doc-chat-eval/vectorstore'
if not os.path.exists(chroma_persist_directory):
    os.makedirs(chroma_persist_directory)
user_id = 'single_topic_rag_evaluation'
vector_store = MultiTenantVectorStore(chroma_persist_directory=chroma_persist_directory, embedding_model='text-embedding-3-small')

Using Chroma persist directory: /tmp/doc-chat-eval/vectorstore


In [16]:
# Index the documents
for index, splits in enumerate(all_splits):
      print('indexing document: ', index, 'number of splits: ', len(splits))
      # enable this to index the documents
      # vector_store.create_document_collection(user_id=user_id,
      #                                         collection_id=f'document_{index}',
      #                                         document_splits=splits,
      #                                         file_name=f'document_{index}')

indexing document:  0 number of splits:  13
indexing document:  1 number of splits:  4
indexing document:  2 number of splits:  29
indexing document:  3 number of splits:  24
indexing document:  4 number of splits:  16
indexing document:  5 number of splits:  21
indexing document:  6 number of splits:  51
indexing document:  7 number of splits:  26
indexing document:  8 number of splits:  16
indexing document:  9 number of splits:  17
indexing document:  10 number of splits:  14
indexing document:  11 number of splits:  86
indexing document:  12 number of splits:  25
indexing document:  13 number of splits:  65
indexing document:  14 number of splits:  24
indexing document:  15 number of splits:  42
indexing document:  16 number of splits:  299
indexing document:  17 number of splits:  31
indexing document:  18 number of splits:  72
indexing document:  19 number of splits:  100


In [17]:
# create chain for retrieval
retrieval_chain = CitationRetrievalChain(retriever=None,
                                         # TODO: use vector store as retriever
                                         llm=create_llm())

In [28]:
# create function to retrieve documents and answer questions with the chain
def retrieve_and_answer(data):
    for index, question_row in data.iterrows():
        question = question_row['question']
        document_index = question_row['document_index']
        collection_id = f'document_{document_index}'

        print(f'Processing question: {index}, collection_id: {collection_id}')

        # Retrieve documents from the vector store
        found_documents = vector_store.retrieve_documents(user_id=user_id, collection_id=collection_id, query=question, k=5)

        # generate answer using the retrieval chain
        response = retrieval_chain.invoke(query=question, documents=found_documents, include_references_in_answer=False)

        # store the answer in the dataframe
        data.loc[index, 'generated_answer'] = response['answer']
        data.loc[index, 'referenced_documents'] = json.dumps(response['referenced_documents'])

    return data



In [29]:
# single passage questions
pd_single_passage = pd.read_csv('../../data/single_topic_rag_evaluation_dataset/processed/single_passage_answer_questions.csv')
pd_result = retrieve_and_answer(pd_single_passage)
pd_result.to_csv('../../data/single_topic_rag_evaluation_dataset/results/single_passage_answer_questions_predict.csv', index=False)

# multi passage questions
pd_multi_passage = pd.read_csv('../../data/single_topic_rag_evaluation_dataset/processed/multi_passage_answer_questions.csv')
pd_result = retrieve_and_answer(pd_multi_passage)
pd_multi_passage.to_csv('../../data/single_topic_rag_evaluation_dataset/results/multi_passage_answer_questions_predict.csv', index=False)

# no answer questions
pd_no_answer = pd.read_csv('../../data/single_topic_rag_evaluation_dataset/processed/no_answer_questions.csv')
pd_result = retrieve_and_answer(pd_no_answer)
pd_no_answer.to_csv('../../data/single_topic_rag_evaluation_dataset/results/no_answer_questions_predict.csv', index=False)

Processing question: 0, collection_id: document_0
Processing question: 1, collection_id: document_0
Processing question: 2, collection_id: document_1


Number of requested results 5 is greater than number of elements in index 4, updating n_results = 4


Processing question: 3, collection_id: document_1


Number of requested results 5 is greater than number of elements in index 4, updating n_results = 4


Processing question: 4, collection_id: document_2
Processing question: 5, collection_id: document_2
Processing question: 6, collection_id: document_3
Processing question: 7, collection_id: document_3
Processing question: 8, collection_id: document_4
Processing question: 9, collection_id: document_4
Processing question: 10, collection_id: document_5
Processing question: 11, collection_id: document_5
Processing question: 12, collection_id: document_6
Processing question: 13, collection_id: document_6
Processing question: 14, collection_id: document_7
Processing question: 15, collection_id: document_7
Processing question: 16, collection_id: document_8
Processing question: 17, collection_id: document_8
Processing question: 18, collection_id: document_9
Processing question: 19, collection_id: document_9
Processing question: 20, collection_id: document_10
Processing question: 21, collection_id: document_10
Processing question: 22, collection_id: document_11
Processing question: 23, collectio

...

Processing question: 3, collection_id: document_1
Processing question: 4, collection_id: document_2
Processing question: 5, collection_id: document_2
Processing question: 6, collection_id: document_3
Processing question: 7, collection_id: document_3
Processing question: 8, collection_id: document_4
Processing question: 9, collection_id: document_4
Processing question: 10, collection_id: document_5
Processing question: 11, collection_id: document_5
Processing question: 12, collection_id: document_6
Processing question: 13, collection_id: document_6
Processing question: 14, collection_id: document_7
Processing question: 15, collection_id: document_7
Processing question: 16, collection_id: document_8
Processing question: 17, collection_id: document_8
Processing question: 18, collection_id: document_9
Processing question: 19, collection_id: document_9
Processing question: 20, collection_id: document_10
Processing question: 21, collection_id: document_10
Processing question: 22, collection_

Number of requested results 5 is greater than number of elements in index 4, updating n_results = 4


Processing question: 2, collection_id: document_1


Number of requested results 5 is greater than number of elements in index 4, updating n_results = 4


Processing question: 3, collection_id: document_1
Processing question: 4, collection_id: document_2
Processing question: 5, collection_id: document_2
Processing question: 6, collection_id: document_3
Processing question: 7, collection_id: document_3
Processing question: 8, collection_id: document_4
Processing question: 9, collection_id: document_4
Processing question: 10, collection_id: document_5
Processing question: 11, collection_id: document_5
Processing question: 12, collection_id: document_6
Processing question: 13, collection_id: document_6
Processing question: 14, collection_id: document_7
Processing question: 15, collection_id: document_7
Processing question: 16, collection_id: document_8
Processing question: 17, collection_id: document_8
Processing question: 18, collection_id: document_9
Processing question: 19, collection_id: document_9
Processing question: 20, collection_id: document_10
Processing question: 21, collection_id: document_10
Processing question: 22, collection_